In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00001.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00002.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00003.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00004.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00005.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00006.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00007.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00008.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00009.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00010.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00011.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00012.jpg
/kaggle/input/datasets/arunrk7/surface-crack-detection/Negative/00013.jpg
/kaggle/input/datasets/arunrk7/surface

In [2]:
import os
print(os.listdir("/kaggle/input/datasets/daikhiyassine"))
print(os.listdir("/kaggle/input/datasets"))

['reinforced-concrete-structure-segmentation-1841', 'mdmcs-dataset', 'reinforced-concrete-structure-segmentation-2119', 'dataset']
['arunrk7', 'daikhiyassine']


In [3]:
import os
import json
from PIL import Image

PADDING_RATIO = 0.15
MIN_CROP_SIZE = 40

def extract_crops_labelme(img_dir, json_dir, mapping, source_tag, output_base):
    """Extraction générique pour annotations LabelMe (shapes/points/label)."""
    stats = {cls: 0 for cls in set(mapping.values())}
    stats["exclus"] = 0
    skipped_too_small = 0
    missing_image = 0
    skipped_invalid_box = 0

    # Index des images par nom de base, insensible à l'extension exacte
    img_index = {}
    for fname in os.listdir(img_dir):
        base = os.path.splitext(fname)[0]
        img_index[base] = fname

    for json_name in os.listdir(json_dir):
        if not json_name.endswith(".json"):
            continue
        base_name = os.path.splitext(json_name)[0]

        if base_name not in img_index:
            missing_image += 1
            continue

        img_path = os.path.join(img_dir, img_index[base_name])
        json_path = os.path.join(json_dir, json_name)

        with open(json_path) as f:
            try:
                annotation = json.load(f)
            except Exception:
                continue

        shapes = annotation.get("shapes", [])

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            continue

        img_w, img_h = img.size

        for shape_idx, shape in enumerate(shapes):
            label = shape.get("label", "").lower().strip()
            target_class = mapping.get(label)
            if target_class is None:
                stats["exclus"] += 1
                continue

            points = shape.get("points", [])
            if len(points) < 2:
                continue

            xs = [p[0] for p in points]
            ys = [p[1] for p in points]
            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)
            box_w, box_h = x_max - x_min, y_max - y_min

            if box_w < MIN_CROP_SIZE or box_h < MIN_CROP_SIZE:
                skipped_too_small += 1
                continue

            pad_x, pad_y = box_w * PADDING_RATIO, box_h * PADDING_RATIO
            crop_x_min = max(0, x_min - pad_x)
            crop_y_min = max(0, y_min - pad_y)
            crop_x_max = min(img_w, x_max + pad_x)
            crop_y_max = min(img_h, y_max + pad_y)

            if crop_x_max <= crop_x_min or crop_y_max <= crop_y_min:
                skipped_invalid_box += 1
                continue

            try:
                crop = img.crop((crop_x_min, crop_y_min, crop_x_max, crop_y_max))
            except Exception:
                skipped_invalid_box += 1
                continue

            out_dir = os.path.join(output_base, target_class)
            os.makedirs(out_dir, exist_ok=True)
            out_name = f"{source_tag}_{base_name}_shape{shape_idx}.jpg"
            crop.save(os.path.join(out_dir, out_name), "JPEG", quality=90)
            stats[target_class] += 1

    stats["images_manquantes"] = missing_image
    stats["crops_trop_petits"] = skipped_too_small
    stats["boites_invalides"] = skipped_invalid_box
    return stats

print("✅ Fonction d'extraction prête")

✅ Fonction d'extraction prête


In [4]:
CROPS_OUTPUT = "/kaggle/working/all_crops_staging"

# --- MDMCS ---
MDMCS_BASE = "/kaggle/input/datasets/daikhiyassine/mdmcs-dataset/MDMCS A Benchmark Dataset for Multi-Damage Monitor/HRCDS/HRCDS"
MDMCS_MAPPING = {"spalling": "spalling", "corrosion": "corrosion", "crack": "cracks"}

print("=== Extraction MDMCS (train/val/test) ===")
mdmcs_total = {"corrosion": 0, "spalling": 0, "cracks": 0}
for split in ["train", "val", "test"]:
    img_dir = os.path.join(MDMCS_BASE, f"{split}_image")
    ann_dir = os.path.join(MDMCS_BASE, f"{split}_annotations")
    if not os.path.isdir(ann_dir):
        continue
    s = extract_crops_labelme(img_dir, ann_dir, MDMCS_MAPPING, f"mdmcs{split}", CROPS_OUTPUT)
    print(f"  {split}:", s)
    for k in mdmcs_total:
        mdmcs_total[k] += s.get(k, 0)
print("Total MDMCS :", mdmcs_total)

# --- RCS 2119 ---
RCS_2119_BASE = "/kaggle/input/datasets/daikhiyassine/reinforced-concrete-structure-segmentation-2119"
MAPPING_2119 = {"crack": "cracks", "rebar corrosion": "corrosion", "concrete spalling": "spalling"}

print("\n=== Extraction RCS 2119 ===")
stats_2119 = extract_crops_labelme(
    os.path.join(RCS_2119_BASE, "imagedata"),
    os.path.join(RCS_2119_BASE, "jsondata"),
    MAPPING_2119, "rcs2119", CROPS_OUTPUT
)
print(stats_2119)

# --- RCS 1841 ---
RCS_1841_BASE = "/kaggle/input/datasets/daikhiyassine/reinforced-concrete-structure-segmentation-1841"
MAPPING_1841 = {
    "structural crack": "cracks", "microcrack": "cracks",
    "rebar corrosion": "corrosion",
    "minor spalling": "spalling", "moderate spalling": "spalling"
}

print("\n=== Extraction RCS 1841 ===")
stats_1841 = extract_crops_labelme(
    os.path.join(RCS_1841_BASE, "imagedata"),
    os.path.join(RCS_1841_BASE, "labeldata"),
    MAPPING_1841, "rcs1841", CROPS_OUTPUT
)
print(stats_1841)

print(f"\n✅ Tous les crops sauvegardés dans : {CROPS_OUTPUT}")

=== Extraction MDMCS (train/val/test) ===
  train: {'spalling': 1239, 'cracks': 282, 'corrosion': 406, 'exclus': 1392, 'images_manquantes': 0, 'crops_trop_petits': 723, 'boites_invalides': 0}
  val: {'spalling': 108, 'cracks': 21, 'corrosion': 26, 'exclus': 102, 'images_manquantes': 0, 'crops_trop_petits': 25, 'boites_invalides': 0}
  test: {'spalling': 82, 'cracks': 26, 'corrosion': 18, 'exclus': 84, 'images_manquantes': 0, 'crops_trop_petits': 2, 'boites_invalides': 0}
Total MDMCS : {'corrosion': 450, 'spalling': 1429, 'cracks': 329}

=== Extraction RCS 2119 ===


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/daikhiyassine/reinforced-concrete-structure-segmentation-2119/imagedata'

In [ ]:
RCS_2119_BASE = "/kaggle/input/datasets/daikhiyassine/reinforced-concrete-structure-segmentation-2119"
MAPPING_2119 = {"crack": "cracks", "rebar corrosion": "corrosion", "concrete spalling": "spalling"}

# Vérifie d'abord le nom exact du dossier avant de lancer
print(os.listdir(RCS_2119_BASE))

In [ ]:
print("\n=== Extraction RCS 2119 ===")
stats_2119 = extract_crops_labelme(
    os.path.join(RCS_2119_BASE, "imagedata (1)"),  # ⚠️ nom exact à confirmer avec le print ci-dessus
    os.path.join(RCS_2119_BASE, "jsondata"),
    MAPPING_2119, "rcs2119", CROPS_OUTPUT
)
print(stats_2119)

In [ ]:
RCS_1841_BASE = "/kaggle/input/datasets/daikhiyassine/reinforced-concrete-structure-segmentation-1841"
print(os.listdir(RCS_1841_BASE))  # vérifie aussi les noms exacts ici, au cas où

In [ ]:
MAPPING_1841 = {
    "structural crack": "cracks", "microcrack": "cracks",
    "rebar corrosion": "corrosion",
    "minor spalling": "spalling", "moderate spalling": "spalling"
}

print("\n=== Extraction RCS 1841 ===")
stats_1841 = extract_crops_labelme(
    os.path.join(RCS_1841_BASE, "imagedata"),   # ⚠️ ajuste si le print montre un nom différent (ex: "imagedata (1)")
    os.path.join(RCS_1841_BASE, "labeldata"),
    MAPPING_1841, "rcs1841", CROPS_OUTPUT
)
print(stats_1841)

In [ ]:
import hashlib
import random
import shutil
from collections import defaultdict

EXISTING_DATASET_PATH = "/kaggle/input/datasets/daikhiyassine/dataset"
SFC_PATH = "/kaggle/input/datasets/arunrk7/surface-crack-detection"
CROPS_STAGING = "/kaggle/working/all_crops_staging"

MERGED_OUTPUT_PATH = "/kaggle/working/dataset_v3_final"

SPLITS = ["train", "val", "test"]
CLASSES = ["corrosion", "spalling", "normal", "cracks"]
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

TARGET_SIZE = 256
JPEG_QUALITY = 85
random.seed(42)

def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# ============================================================
# 1️⃣ Indexer les hashs du dataset existant
# ============================================================
print("Étape 1/7 : indexation du dataset existant...")

existing_hashes = set()
existing_counts = {split: {cls: 0 for cls in CLASSES} for split in SPLITS}

for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(EXISTING_DATASET_PATH, split, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            try:
                existing_hashes.add(get_file_hash(img_path))
                existing_counts[split][cls] += 1
            except Exception:
                continue

total_existing = sum(sum(c.values()) for c in existing_counts.values())
print(f"✅ {len(existing_hashes)} images indexées")
if total_existing == 0:
    print("🚨 ERREUR : vérifie EXISTING_DATASET_PATH avant de continuer !")

# ============================================================
# 2️⃣ Collecter TOUTES les nouvelles images (crops + Surface Crack Detection)
# ============================================================
print("\nÉtape 2/7 : collecte de toutes les nouvelles images...")

new_images_by_class = defaultdict(list)

# --- Crops (MDMCS + RCS 2119 + RCS 1841), déjà en dossiers par classe ---
for cls in ["corrosion", "spalling", "cracks"]:
    cls_dir = os.path.join(CROPS_STAGING, cls)
    if os.path.isdir(cls_dir):
        for img_name in os.listdir(cls_dir):
            # le tag de source est déjà dans le nom du fichier (mdmcs.../rcs2119.../rcs1841...)
            source_tag = img_name.split("_")[0]
            new_images_by_class[cls].append((os.path.join(cls_dir, img_name), source_tag))

# --- Surface Crack Detection (cracks/normal) ---
sfc_mapping = {"Positive": "cracks", "Negative": "normal"}
for source_folder, target_class in sfc_mapping.items():
    folder_path = os.path.join(SFC_PATH, source_folder)
    if not os.path.isdir(folder_path):
        print(f"⚠️ Dossier introuvable : {folder_path}")
        continue
    for img_name in os.listdir(folder_path):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            new_images_by_class[target_class].append((os.path.join(folder_path, img_name), "sfc"))

for cls, imgs in new_images_by_class.items():
    print(f"   {cls}: {len(imgs)} images (toutes sources)")

# ============================================================
# 3️⃣ Déduplication (contre existant ET entre toutes les nouvelles sources)
# ============================================================
print("\nÉtape 3/7 : déduplication...")

unique_new_by_class = defaultdict(list)
stats_dedup = {"doublons_avec_existant": 0, "doublons_internes": 0}
seen_hashes_internal = set()

for cls, img_entries in new_images_by_class.items():
    shuffled = img_entries.copy()
    random.shuffle(shuffled)
    for img_path, tag in shuffled:
        try:
            h = get_file_hash(img_path)
        except Exception:
            continue
        if h in existing_hashes:
            stats_dedup["doublons_avec_existant"] += 1
        elif h in seen_hashes_internal:
            stats_dedup["doublons_internes"] += 1
        else:
            seen_hashes_internal.add(h)
            unique_new_by_class[cls].append((img_path, tag))

print(f"   Doublons avec l'existant : {stats_dedup['doublons_avec_existant']}")
print(f"   Doublons internes (entre sources) : {stats_dedup['doublons_internes']}")
for cls, imgs in unique_new_by_class.items():
    print(f"   ✅ {cls}: {len(imgs)} images uniques disponibles")

# ============================================================
# 4️⃣ Split train/val/test — on garde TOUT pour corrosion/spalling,
#     plafond raisonnable pour cracks/normal (déjà abondants)
# ============================================================
print("\nÉtape 4/7 : split train/val/test...")

CAPS = {"corrosion": None, "spalling": None, "cracks": 5000, "normal": 5000}
new_split_assignment = defaultdict(lambda: defaultdict(list))
sampling_stats = {}

for cls, img_entries in unique_new_by_class.items():
    n_available = len(img_entries)
    cap = CAPS.get(cls)
    n_target = min(cap, n_available) if cap else n_available
    sampled = random.sample(img_entries, n_target)

    source_breakdown = defaultdict(int)
    for _, tag in sampled:
        source_breakdown[tag] += 1
    sampling_stats[cls] = {"disponibles": n_available, "retenues": n_target, "sources": dict(source_breakdown)}

    n_train = int(n_target * SPLIT_RATIOS["train"])
    n_val = int(n_target * SPLIT_RATIOS["val"])

    new_split_assignment["train"][cls] = sampled[:n_train]
    new_split_assignment["val"][cls] = sampled[n_train:n_train + n_val]
    new_split_assignment["test"][cls] = sampled[n_train + n_val:]

    print(f"   {cls}: {n_target}/{n_available} retenues — sources: {dict(source_breakdown)}")

# ============================================================
# 5️⃣ Construction du dataset final
# ============================================================
print(f"\nÉtape 5/7 : construction dans {MERGED_OUTPUT_PATH}...")

def process_and_save(src_path, dst_path):
    with Image.open(src_path) as img:
        if img.mode != "RGB":
            img = img.convert("RGB")
        img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
        img.save(dst_path, "JPEG", quality=JPEG_QUALITY, optimize=True)

copied_v1 = 0
for split in SPLITS:
    for cls in CLASSES:
        src_dir = os.path.join(EXISTING_DATASET_PATH, split, cls)
        dst_dir = os.path.join(MERGED_OUTPUT_PATH, split, cls)
        os.makedirs(dst_dir, exist_ok=True)
        if not os.path.isdir(src_dir):
            continue
        for img_name in os.listdir(src_dir):
            try:
                shutil.copy2(os.path.join(src_dir, img_name), os.path.join(dst_dir, img_name))
                copied_v1 += 1
            except Exception as e:
                print(f"⚠️ {e}")

print(f"   {copied_v1} images v1 copiées")

added_v2 = 0
for split in SPLITS:
    for cls in CLASSES:
        dst_dir = os.path.join(MERGED_OUTPUT_PATH, split, cls)
        os.makedirs(dst_dir, exist_ok=True)
        for img_path, tag in new_split_assignment[split].get(cls, []):
            fname = os.path.basename(img_path)
            base_name = os.path.splitext(fname)[0]
            dst_path = os.path.join(dst_dir, f"{base_name}.jpg")
            try:
                process_and_save(img_path, dst_path)
                added_v2 += 1
            except Exception as e:
                print(f"⚠️ {e}")

print(f"   {added_v2} nouvelles images ajoutées")

# ============================================================
# 6️⃣ Déduplication finale de sécurité
# ============================================================
print("\nÉtape 6/7 : déduplication finale de sécurité...")

hash_map_final = defaultdict(list)
for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(MERGED_OUTPUT_PATH, split, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            h = get_file_hash(img_path)
            hash_map_final[h].append(img_path)

removed_final = 0
for h, paths in hash_map_final.items():
    if len(paths) > 1:
        for p in paths[1:]:
            os.remove(p)
            removed_final += 1

print(f"   {removed_final} doublons résiduels supprimés")

# ============================================================
# 7️⃣ Documentation
# ============================================================
print("\nÉtape 7/7 : documentation...")

final_counts = {}
for split in SPLITS:
    final_counts[split] = {}
    for cls in CLASSES:
        cls_path = os.path.join(MERGED_OUTPUT_PATH, split, cls)
        final_counts[split][cls] = len(os.listdir(cls_path)) if os.path.isdir(cls_path) else 0

total_v1 = sum(sum(c.values()) for c in existing_counts.values())
total_final = sum(sum(c.values()) for c in final_counts.values())

rapport = {
    "date_fusion": "10/08/2026",
    "sources": [
        {"nom": "MDMCS v2", "licence": "CC BY 4.0", "doi": "10.17632/6x4dzzrs2h.2"},
        {"nom": "Reinforced Concrete Structure Segmentation (2119)", "licence": "CC BY 4.0", "doi": "10.17632/2vkm6k4cfg.1"},
        {"nom": "Reinforced Concrete Structure Segmentation (1841)", "licence": "CC BY 4.0", "doi": "10.17632/yfhwfcrfmk.1"},
        {"nom": "Surface Crack Detection (Özgenel)", "licence": "CC BY 4.0", "doi": "10.17632/5y9wdsg2zt.2"}
    ],
    "stats_echantillonnage": sampling_stats,
    "stats_deduplication": stats_dedup,
    "doublons_residuels_supprimes": removed_final,
    "images_v1_original": copied_v1,
    "images_nouvelles_nettes": added_v2 - removed_final,
    "images_total_final": total_v1 + added_v2 - removed_final,
    "comparaison_v1_vs_final": {"v1": existing_counts, "final": final_counts}
}

with open(os.path.join(MERGED_OUTPUT_PATH, "rapport_fusion_v3.json"), "w") as f:
    json.dump(rapport, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("RÉSUMÉ FINAL — DATASET V3")
print("=" * 60)
print(f"v1 (original) : {total_v1} images")
print(f"Nouvelles nettes : {added_v2 - removed_final}")
print(f"TOTAL FINAL : {total_v1 + added_v2 - removed_final} images")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        v1_n = existing_counts[split][cls]
        v2_n = final_counts[split][cls]
        print(f"   {cls}: {v1_n} → {v2_n} ({'+' if v2_n-v1_n>0 else ''}{v2_n-v1_n})")

print(f"\n✅ Dataset final : {MERGED_OUTPUT_PATH}")

In [ ]:
import os

RCS_1841_BASE = "/kaggle/input/datasets/daikhiyassine/reinforced-concrete-structure-segmentation-1841"
print(os.listdir(RCS_1841_BASE))  # vérifie les noms de dossiers avant de lancer

MAPPING_1841 = {
    "structural crack": "cracks", "microcrack": "cracks",
    "rebar corrosion": "corrosion",
    "minor spalling": "spalling", "moderate spalling": "spalling"
}

CROPS_OUTPUT = "/kaggle/working/all_crops_staging"  # même dossier que les extractions précédentes

print("\n=== Extraction RCS 1841 ===")
stats_1841 = extract_crops_labelme(
    os.path.join(RCS_1841_BASE, "imagedata"),
    os.path.join(RCS_1841_BASE, "labeldata"),
    MAPPING_1841, "rcs1841", CROPS_OUTPUT
)
print(stats_1841)

In [ ]:
import hashlib
import random
from collections import defaultdict
from PIL import Image
import json

MERGED_OUTPUT_PATH = "/kaggle/working/dataset_v3_final"
SPLITS = ["train", "val", "test"]
CLASSES = ["corrosion", "spalling", "normal", "cracks"]
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
TARGET_SIZE = 256
JPEG_QUALITY = 85
random.seed(43)  # seed différente pour ce lot additionnel

def get_file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

# 1. Indexer les hashs du dataset v3 déjà construit (état actuel)
print("Indexation du dataset v3 actuel...")
current_hashes = set()
current_counts = {split: {cls: 0 for cls in CLASSES} for split in SPLITS}
for split in SPLITS:
    for cls in CLASSES:
        cls_path = os.path.join(MERGED_OUTPUT_PATH, split, cls)
        if not os.path.isdir(cls_path):
            continue
        for img_name in os.listdir(cls_path):
            img_path = os.path.join(cls_path, img_name)
            current_hashes.add(get_file_hash(img_path))
            current_counts[split][cls] += 1
print(f"✅ {len(current_hashes)} images déjà présentes dans dataset_v3_final")

# 2. Collecter les nouveaux crops RCS 1841 uniquement
rcs1841_by_class = defaultdict(list)
for cls in ["corrosion", "spalling", "cracks"]:
    cls_dir = os.path.join(CROPS_OUTPUT, cls)
    if os.path.isdir(cls_dir):
        for img_name in os.listdir(cls_dir):
            if img_name.startswith("rcs1841_"):
                rcs1841_by_class[cls].append(os.path.join(cls_dir, img_name))

for cls, imgs in rcs1841_by_class.items():
    print(f"   RCS1841 - {cls}: {len(imgs)} images trouvées")

# 3. Dédupliquer contre l'existant
unique_rcs1841 = defaultdict(list)
dup_count = 0
for cls, img_paths in rcs1841_by_class.items():
    for img_path in img_paths:
        h = get_file_hash(img_path)
        if h in current_hashes:
            dup_count += 1
        else:
            current_hashes.add(h)  # évite aussi les doublons internes
            unique_rcs1841[cls].append(img_path)

print(f"\nDoublons évités : {dup_count}")
for cls, imgs in unique_rcs1841.items():
    print(f"   ✅ {cls}: {len(imgs)} images nettes à ajouter")

# 4. Split et ajout dans le dataset v3 (pas de plafond, corrosion/spalling prioritaires)
def process_and_save(src_path, dst_path):
    with Image.open(src_path) as img:
        if img.mode != "RGB":
            img = img.convert("RGB")
        img = img.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
        img.save(dst_path, "JPEG", quality=JPEG_QUALITY, optimize=True)

added_count = 0
for cls, img_paths in unique_rcs1841.items():
    shuffled = img_paths.copy()
    random.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * SPLIT_RATIOS["train"])
    n_val = int(n * SPLIT_RATIOS["val"])
    split_assign = {
        "train": shuffled[:n_train],
        "val": shuffled[n_train:n_train + n_val],
        "test": shuffled[n_train + n_val:]
    }
    for split, paths in split_assign.items():
        dst_dir = os.path.join(MERGED_OUTPUT_PATH, split, cls)
        os.makedirs(dst_dir, exist_ok=True)
        for src_path in paths:
            fname = os.path.basename(src_path)
            dst_path = os.path.join(dst_dir, fname)
            try:
                process_and_save(src_path, dst_path)
                added_count += 1
            except Exception as e:
                print(f"⚠️ {e}")

print(f"\n✅ {added_count} images RCS1841 ajoutées au dataset v3")

# 5. Recompte final
final_counts = {}
for split in SPLITS:
    final_counts[split] = {}
    for cls in CLASSES:
        cls_path = os.path.join(MERGED_OUTPUT_PATH, split, cls)
        final_counts[split][cls] = len(os.listdir(cls_path)) if os.path.isdir(cls_path) else 0

total_final = sum(sum(c.values()) for c in final_counts.values())
print(f"\n{'='*60}\nTOTAL DATASET V3 (avec RCS1841) : {total_final} images\n{'='*60}")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        print(f"   {cls}: {final_counts[split][cls]}")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/dataset_v3_final",  # nom du zip (sans extension)
    "zip",
    "/kaggle/working/dataset_v3_final"   # dossier source à compresser
)
print("✅ Zip créé")

In [ ]:
import os
size_gb = os.path.getsize("/kaggle/working/dataset_v3_final.zip") / (1024**3)
print(f"Taille du zip : {size_gb:.2f} Go")

In [ ]:
# Nécessite kaggle.json configuré (Settings > API > Create New Token)
!pip install -q kaggle
import json, os

os.makedirs("/root/.config/kaggle", exist_ok=True)
# place ton kaggle.json ici si besoin, ou utilise les secrets Kaggle

dataset_metadata = {
    "title": "Concrete Damage Dataset v3",
    "id": "TON_USERNAME/concrete-damage-v3",
    "licenses": [{"name": "CC-BY-4.0"}]
}
os.makedirs("/kaggle/working/dataset_upload", exist_ok=True)
with open("/kaggle/working/dataset_upload/dataset-metadata.json", "w") as f:
    json.dump(dataset_metadata, f)

shutil.move("/kaggle/working/dataset_v3_final.zip", "/kaggle/working/dataset_upload/dataset_v3_final.zip")
!kaggle datasets create -p /kaggle/working/dataset_upload

In [ ]:
import json, os

dataset_metadata = {
    "title": "Concrete Damage Dataset v3",
    "id": "ton-vrai-username/concrete-damage-v3",  # remplace ici, tout en minuscules
    "licenses": [{"name": "CC-BY-4.0"}]
}

with open("/kaggle/working/dataset_upload/dataset-metadata.json", "w") as f:
    json.dump(dataset_metadata, f)

!kaggle datasets create -p /kaggle/working/dataset_upload


In [ ]:
!kaggle config view

In [ ]:
import json
with open("/root/.config/kaggle/kaggle.json") as f:
    creds = json.load(f)
print(creds["username"])

In [ ]:
import json

username = "colle_ici_ton_vrai_username"  # remplace par le résultat de l'étape 1

dataset_metadata = {
    "title": "Concrete Damage Dataset v3",
    "id": f"{username}/concrete-damage-v3",
    "licenses": [{"name": "CC-BY-4.0"}]
}

with open("/kaggle/working/dataset_upload/dataset-metadata.json", "w") as f:
    json.dump(dataset_metadata, f)

print(dataset_metadata)  # vérifie que ça affiche bien ton vrai username, pas TON_USERNAME

In [ ]:
!kaggle datasets create -p /kaggle/working/dataset_upload

In [ ]:
import json

dataset_metadata = {
    "title": "Concrete Damage Dataset v3",
    "id": "daikhiyassine/concrete-damage-v3",
    "licenses": [{"name": "CC-BY-4.0"}]
}

with open("/kaggle/working/dataset_upload/dataset-metadata.json", "w") as f:
    json.dump(dataset_metadata, f)

print(dataset_metadata)  # doit afficher "daikhiyassine/concrete-damage-v3"

In [ ]:
!kaggle datasets create -p /kaggle/working/dataset_upload

In [ ]:
import zipfile

with zipfile.ZipFile("/kaggle/working/dataset_upload/dataset_v3_final.zip") as z:
    names = z.namelist()
    print(f"Nombre de fichiers dans le zip : {len(names)}")
    print("Premiers chemins :")
    for n in names[:10]:
        print(" ", n)

In [ ]:
!kaggle datasets status daikhiyassine/concrete-damage-v3

In [ ]:
kaggle datasets download -d daikhiyassine/concrete-damage-v3